# Spatial associations between CD8 and myeloid cells (Figure 3B / Supplementary Figure 4)

To more precisely predict the location of myeloid cells relative to antigen-experienced CD8 subsets, we performed Pearson correlation to measure co-enrichment of cell-types within the same Visium spots (Fig. 3B). These analyses showed correlation of migDC with early active and proliferating CD8s, moDC-B with proliferating CD8s, cDC2 with late active CD8s, and red pulp-localized myeloid cells (F480 macs, cDC1, Mac-A, Mac-B, moDC-A, monocytes) with both CD8 late active and effector populations. Similar relationships between myeloid and CD8 subsets were observed using the alternative spatial autocorrelation method (Fig. S4D).

1. Fig. 3B — within-spot Pearson correlation of cell2location abundances
2. Fig. S4D — global bivariate Moran’s I with inverse-distance spatial weights (9,999 permutations)



## 1. Setup


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from sklearn.neighbors import NearestNeighbors
from libpysal.weights import W
from esda.moran import Moran_BV
from scipy.stats import pearsonr

# Paths (override with env vars or edit here)
INFECTED_H5AD = Path(
    os.environ.get(
        "C2L_INFECTED_H5AD",
        "data/cell2location_infected.h5ad",
    )
)
OUT_DIR = Path(os.environ.get("CD8_MYELOID_OUT", "outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

ABUNDANCE_KEY = "means_cell_abundance_w_sf"
SAMPLES = ["V1S1", "V1S2"]  # infected Visium sections
RADIUS = 300  # coordinate units for neighbor graph
PERMUTATIONS = 9999

CONVENTIONAL_CD8 = [
    "CD8-Tcell_naive",
    "CD8-Tcell_early-active",
    "CD8-Tcell_effector",
    "CD8-Tcell_late-active",
    "CD8-Tcell_terminal",
    "CD8-Tcell_proliferating",
]

# Display / row-column order for paper-style heatmaps
CD8_DISPLAY = {
    "CD8-Tcell_naive": "naive",
    "CD8-Tcell_early-active": "early-active",
    "CD8-Tcell_proliferating": "proliferating",
    "CD8-Tcell_late-active": "late-active",
    "CD8-Tcell_effector": "effector",
}
CD8_ROW_ORDER = list(CD8_DISPLAY.keys())

MYELOID_DISPLAY = {
    "Myeloid_migratory": "MigDC",
    "Myeloid_pDC": "pDC",
    "Myeloid_moDC-CXCL9/10": "moDC-B",
    "Myeloid_DC2": "cDC2",
    "Myeloid_cDC1": "cDC1",
    "Myeloid_moDC": "moDC-A",
    "Myeloid_Macrophage": "Mac-A",
    "Myeloid_Macrophage-CXCL9/10": "Mac-B",
    "Myeloid_marginalzone": "Mac-F480",
    "Myeloid_monocyte": "Monocyte",
}
MYELOID_COL_ORDER = list(MYELOID_DISPLAY.values())

adata = sc.read_h5ad(INFECTED_H5AD)
ab = adata.obsm[ABUNDANCE_KEY]
if not isinstance(ab, pd.DataFrame):
    ab = pd.DataFrame(ab, index=adata.obs_names)
adata.obsm[ABUNDANCE_KEY] = ab

myeloid_types = [c for c in ab.columns if "Myeloid" in str(c)]
print(adata.n_obs, "spots |", len(myeloid_types), "myeloid columns | samples:", SAMPLES)


## 2. Within-spot Pearson correlation (Figure 3B)

Pearson *r* between CD8 and myeloid abundances across spots (infected sections combined).


In [ ]:
pearson_mat = pd.DataFrame(index=CD8_ROW_ORDER, columns=list(MYELOID_DISPLAY.keys()), dtype=float)

for cd8 in CD8_ROW_ORDER:
    for myel in MYELOID_DISPLAY:
        if cd8 not in ab.columns or myel not in ab.columns:
            continue
        x = ab[cd8].astype(float).values
        y = ab[myel].astype(float).values
        pearson_mat.loc[cd8, myel] = pearsonr(x, y)[0] if x.std() and y.std() else np.nan

pearson_plot = pearson_mat.rename(index=CD8_DISPLAY, columns=MYELOID_DISPLAY)
pearson_plot = pearson_plot.reindex(index=list(CD8_DISPLAY.values()), columns=MYELOID_COL_ORDER)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.heatmap(
    pearson_plot.astype(float),
    cmap="RdBu_r",
    center=0,
    vmin=-0.5,
    vmax=0.8,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Pearson r"},
    ax=ax,
)
ax.set_title("Fig. 3B - within-spot CD8 vs myeloid (infected)")
ax.set_xlabel("Myeloid")
ax.set_ylabel("CD8")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
fig.savefig(OUT_DIR / "fig3b_cd8_myeloid_pearson_infected.pdf", bbox_inches="tight")
fig.savefig(OUT_DIR / "fig3b_cd8_myeloid_pearson_infected.png", dpi=300, bbox_inches="tight")
plt.show()
pearson_mat.to_csv(OUT_DIR / "fig3b_cd8_myeloid_pearson_infected.csv")
print("Saved Pearson heatmap + CSV →", OUT_DIR)


## 3. Distance-based spatial weights

Inverse-distance weights within `RADIUS` coordinate units. Spots with no neighbors are excluded from Moran's I.


In [ ]:
def create_distance_based_weights(coords, radius=RADIUS):
    nbrs = NearestNeighbors(radius=radius, algorithm="ball_tree").fit(coords)
    distances, indices = nbrs.radius_neighbors(coords)

    neighbors, weights_dict = {}, {}
    for i in range(coords.shape[0]):
        mask = indices[i] != i
        neighbor_idx = indices[i][mask]
        neighbor_dist = distances[i][mask]
        if len(neighbor_idx) == 0:
            continue
        weights = 1.0 / (neighbor_dist + 1e-10)
        neighbors[i] = neighbor_idx.tolist()
        weights_dict[i] = weights.tolist()
    return W(neighbors, weights_dict)


weights_by_sample = {}
indices_by_sample = {}

for sample in SAMPLES:
    mask = (adata.obs["sample"] == sample).values
    indices = np.where(mask)[0]
    coords = adata.obsm["spatial"][mask]
    w = create_distance_based_weights(coords, radius=RADIUS)
    weights_by_sample[sample] = w
    indices_by_sample[sample] = indices
    n_isolated = sum(1 for i in range(len(indices)) if i not in w.neighbors)
    print(
        f"{sample}: {len(indices)} spots, {w.n} connected, "
        f"{n_isolated} isolated, mean neighbors={w.mean_neighbors:.1f}"
    )


## 4. Global bivariate Moran's I (S4)

One *I* and pseudo-*P* per CD8-myeloid pair per sample (`PERMUTATIONS` permutations).


In [ ]:
rows = []
for sample in SAMPLES:
    indices = indices_by_sample[sample]
    abundances = adata.obsm[ABUNDANCE_KEY].iloc[indices]
    w = weights_by_sample[sample]
    connected = list(w.neighbors.keys())
    ab_conn = abundances.iloc[connected]

    print(f"{sample}:")
    for cd8 in CONVENTIONAL_CD8:
        for myel in myeloid_types:
            if cd8 not in ab_conn.columns or myel not in ab_conn.columns:
                continue
            x = ab_conn[cd8].values.astype(np.float64)
            y = ab_conn[myel].values.astype(np.float64)
            if x.std() == 0 or y.std() == 0:
                continue
            moran_bv = Moran_BV(x, y, w, permutations=PERMUTATIONS)
            rows.append(
                {
                    "sample": sample,
                    "condition": "infected",
                    "cd8_type": cd8,
                    "myeloid_type": myel,
                    "I": moran_bv.I,
                    "p_value": moran_bv.p_sim,
                    "n_spots": len(connected),
                }
            )
    print(f"  {sum(1 for r in rows if r['sample'] == sample)} pairs")

moran_results = pd.DataFrame(rows)
moran_results.to_csv(OUT_DIR / "global_bivariate_moran_infected.csv", index=False)
print(f"Saved {len(moran_results)} rows →", OUT_DIR / "global_bivariate_moran_infected.csv")


## 5. Mean across infected replicates + heatmap

Average *I* and pseudo-*P* across sections; gray-mask cells with mean *P* ≥ 0.05.


In [ ]:
mean_values = (
    moran_results.groupby(["cd8_type", "myeloid_type"])
    .agg(I=("I", "mean"), p_value=("p_value", "mean"))
    .reset_index()
)

I_mat = mean_values.pivot(index="cd8_type", columns="myeloid_type", values="I")
P_mat = mean_values.pivot(index="cd8_type", columns="myeloid_type", values="p_value")

I_mat = I_mat.reindex(CD8_ROW_ORDER)
P_mat = P_mat.reindex(CD8_ROW_ORDER)

keep_cols = [c for c in MYELOID_DISPLAY if c in I_mat.columns]
I_mat = I_mat[keep_cols].rename(index=CD8_DISPLAY, columns=MYELOID_DISPLAY)
P_mat = P_mat[keep_cols].rename(index=CD8_DISPLAY, columns=MYELOID_DISPLAY)
I_mat = I_mat.reindex(index=list(CD8_DISPLAY.values()), columns=MYELOID_COL_ORDER)
P_mat = P_mat.reindex(index=list(CD8_DISPLAY.values()), columns=MYELOID_COL_ORDER)

ns_mask = P_mat >= 0.05

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.heatmap(
    I_mat.astype(float),
    cmap="RdBu_r",
    center=0,
    vmin=-0.5,
    vmax=0.8,
    mask=ns_mask,
    cbar_kws={"label": "Bivariate Moran's I"},
    linewidths=0.5,
    linecolor="white",
    ax=ax,
)
if ns_mask.to_numpy().any():
    sns.heatmap(
        I_mat.astype(float),
        mask=~ns_mask,
        cmap=ListedColormap(["#D3D3D3"]),
        cbar=False,
        linewidths=0.5,
        linecolor="white",
        ax=ax,
    )

ax.set_title("S4 - bivariate Moran's I (infected, mean of sections)")
ax.set_xlabel("Myeloid")
ax.set_ylabel("CD8")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
fig.savefig(OUT_DIR / "s4_cd8_myeloid_bivariate_moran_infected.pdf", bbox_inches="tight")
fig.savefig(OUT_DIR / "s4_cd8_myeloid_bivariate_moran_infected.png", dpi=300, bbox_inches="tight")
plt.show()

mean_values.to_csv(OUT_DIR / "global_bivariate_moran_infected_mean.csv", index=False)
print("Saved Moran heatmap + mean table →", OUT_DIR)


## Notes

- Source WIP: `wip/colocalization_myeloid_cd8_12026.ipynb` (and Moran cell in the paper heatmaps notebook).
- `wip/differential_spatial_colocalization_analysis.ipynb` has exploratory follow-ups and unrelated expression panels - left untouched.
- Requires `esda`, `libpysal`, `scanpy`, `seaborn`.
